# Assignment 2

**Credits**: Federico Ruggeri, Giulia Grundler, Paolo Torroni

**Keywords**: Fallacy Detection, Multi-label Classification, LLMs, Prompting

# Contact

For any doubt, question, issue or help, you can always contact us at the following email addresses:

Teaching Assistants:

* Federico Ruggeri -> federico.ruggeri6@unibo.it
* Giulia Grundler -> giulia.grundler2@unibo.it

Professor:

* Paolo Torroni -> p.torroni@unibo.it

# Relevant Material

- Huggingface documentation
- Huggingface hub

# Introduction

You are tasked to address **fallacy detection** on
[MAFALDA](https://github.com/ChadiHelwe/MAFALDA), a benchmark that unifies several older fallacy corpora under a
single taxonomy.

A **fallacy** is an argument whose premises do not entail its conclusion. Fallacies are everywhere in online
discussion, and spotting them is a task both humans and models find hard.

## Problem definition

Given an input text, the task is to say which **categories of fallacy** it contains, out of three:

| Label | Category | Aristotle | Meaning |
|-------|----------|-----------|---------|
| `credibility` | Fallacy of credibility | ethos | The argument leans on who is speaking, or on who else agrees, instead of on evidence. |
| `logic` | Fallacy of logic | logos | The reasoning itself is broken: the conclusion does not follow from the premises. |
| `emotion` | Appeal to emotion | pathos | The argument works on the reader's feelings instead of on the merits of the claim. |

A text may contain **none** of them, **one**, or **several** at once, so this is a **multi-label** problem.

MAFALDA also annotates *where* each fallacy occurs and *which* of 23 fine-grained types it is. We drop both: you
classify the whole text, and only at the level of the three categories above. The fine-grained level is left for the
bonus points.

### Examples

All the examples below are taken from the corpus.

**Text**: *``Two of my best friends are really introverted, shy people, and they both have cats. That leads to me
believe that most cat lovers are really shy.''*

**Label**: `logic` (a hasty generalization: two friends do not settle the question)

**Text**: *``Four out of five dentists recommend Happy Glossy toothpaste. Therefore, it must be great.''*

**Label**: `credibility` (an appeal to a false authority)

**Text**: *``Sign petition for Persona 5 pc and switch ports. So much important stuff in the world that needs
political activism. Makes a petition to portbeg P5 onto PC and Switch.''*

**Label**: `emotion` (an appeal to a worse problem, and a ridicule of the request)

**Text**: *``I know that our TV advertisements are more effective than radio. The numbers show that we hit twice the
audience with TV, and our focus groups remember the TV commercial 38 percent more than the radio slot.''*

**Label**: none (a conclusion its premises actually support)

### More than one category

''*Men score better on math than women do. Jerry is a man. Therefore, Jerry is better at math than Sylvia, who is
a woman.*'' $\rightarrow$ `logic` alone (a fallacy of division).

''*Brandon: We should have tastier lunches! Jaylen: Don't listen to him! He's a terrible person! I saw him trip
another student and steal his lunch money!*'' $\rightarrow$ `credibility` alone (an ad hominem).

A text arguing from a vague consensus *and* frightening the reader would carry `credibility` **and** `emotion`.

In `a2_test.csv`, 31 texts out of 156 carry two or three categories at once, and 50 carry none.

## Approach

We will tackle the task with LLMs.

In particular, we'll consider zero-/few-shot prompting approaches to assess the capability of some popular
open-source LLMs on this task.

## Preliminaries

We are going to download LLMs from [Huggingface](https://huggingface.co/).

Many of these open-source LLMs require you to accept their "Community License Agreement" to download them.

In summary:

- If not already, create an account of Huggingface (~2 mins)
- Check a LLM model card page (e.g., [Mistral v3](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)) and accept its "Community License Agreement".
- Go to your account -> Settings -> Access Tokens -> Create new token -> "Repositories permissions" -> add the LLM model card you want to use.
- Save the token (we'll need it later)

### Huggingface Login

Once we have created an account and an access token, we need to login to Huggingface via code.

- Type your token and press Enter
- You can say No to Github linking

In [ ]:
!hf auth login

After login, you can download all models associated with your access token in addition to those that are not
protected by an access token.

### Data Loading

Since we are only interested in prompting, we do not require a train dataset.

We have prepared a version of MAFALDA in our dedicated
[Github repository](https://github.com/lt-nlp-lab-unibo/nlp-course-material).

Check the ``Assignment 2/data`` folder.
It contains:

- ``a2_test.csv`` -> a test set of 156 texts.
- ``demonstrations.csv`` -> a pool of 44 texts for few-shot prompting.

Both files have the same four columns:

- ``mafalda_id`` -> the line of the text in the original MAFALDA release.
- ``text`` -> the text to classify.
- ``labels_level1`` -> the categories of the text, semicolon-separated, **empty** when the text has no fallacy.
- ``labels_level2`` -> the fine-grained fallacy types, semicolon-separated. Only needed for the bonus.

### Label distribution

| file | texts | no fallacy | credibility | emotion | logic |
|---|---|---|---|---|---|
| `a2_test.csv` | 156 | 50 | 42 | 28 | 76 |
| `demonstrations.csv` | 44 | 13 | 13 | 10 | 22 |

Columns do not sum to the number of texts: a text can carry several categories.

The corpus is small and `emotion` is the rarest category, so expect noisy numbers. Say so in your report rather than
reading too much into a two-point gap.

### Instructions

We require you to:

* **Download** the ``Assignment 2/data`` folder.
* **Encode** ``a2_test.csv`` into a ``pandas.DataFrame`` object.
* **Convert** ``labels_level1`` into a 3-dimensional binary vector, in the order
  ``[credibility, emotion, logic]``.

# [Task 1 - 0.5 points] Model setup

Once the test data has been loaded, we have to setup the model pipeline for inference.

In particular, we have to:
- Load the model weights from Huggingface
- Quantize the model to fit into a single-GPU limited hardware

## Which LLMs?

The pool of LLMs is ever increasing and it's impossible to keep track of all new entries.

We focus on popular open-source models.

All of the following run on the free Colab T4 once quantized to 4 bits.

| Model card | Params | Note |
|---|---|---|
| [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) | 4.0B | Good starting point. This variant does not emit chains of thought. |
| [Mistral-7B-Instruct-v0.3](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) | 7.2B | |
| [Phi-4-mini-instruct](https://huggingface.co/microsoft/Phi-4-mini-instruct) | 3.8B | |
| [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) | 8.2B | Largest that fits comfortably. Pass `enable_thinking=False` to the chat template. |
| [SmolLM3-3B](https://huggingface.co/HuggingFaceTB/SmolLM3-3B) | 3.1B | |
| [TinyLlama](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) | 1.1B | Weak on purpose: useful to see what a bad fail-ratio looks like. |
| [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) | 8.0B | **Access is granted manually**, so request it days before the deadline. |
| [DeepSeek-R1-Distill-Qwen-7B](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B) | 7.6B | Reasoning model: it writes a long chain of thought before answering, which is slow on a T4 and needs stripping in `process_response`. |

Other open-source models are more than welcome!

### Instructions

In order to get Task 1 points, we require you to:

* Pick 2 model cards from the provided list.
* For each model:
  - Setup a quantization configuration for the model.
  - Load the model via HuggingFace APIs.

### Note

There's a popular library integrated with Huggingface's ``transformers`` to perform quantization.

Two things about the Colab T4 specifically:

* It has **no bfloat16** support, so set the compute dtype of your quantization config to ``float16``.
* It has 15 GB of memory, which an 8B model in half precision does not fit. Quantized to 4 bits it takes about 5 GB,
  and the rest goes to the KV cache.

**Watch the context length**: MAFALDA texts are longer than tweets (71 words on average, up to 269), and few-shot
demonstrations add more. Check the context window of the models you pick.

# [Task 2 - 1.0 points] Prompt setup

Prompting requires an input pre-processing phase where we convert each input example into a specific instruction
prompt.

## Prompt Template

Use the following prompt template to process input texts.

In [ ]:
prompt = [
    {
        'role': 'system',
        'content': 'You are an annotator for fallacy detection.'
    },
    {
        'role': 'user',
        'content': """Your task is to detect fallacies in the input text.
         A fallacy is an argument whose premises do not entail its
         conclusion. Classify the input text according to which of the
         following three categories of fallacy it contains.

         Below you find the category definitions:
         Credibility: the argument relies on who is speaking, or on who
         else agrees, instead of on evidence.
         Logic: the reasoning is broken, the conclusion does not follow
         from the premises.
         Emotion: the argument works on the feelings of the reader
         instead of on the merits of the claim.

         A text may contain more than one category, or none at all.

         Respond only by writing the categories that apply, separated by
         a comma, among the following: credibility, logic, emotion.
         Respond only by writing none if the text contains no fallacy.

        TEXT: {text}

        ANSWER:
        """
    }
]

### Instructions

In order to get Task 2 points, we require you to:

* Write a ``prepare_prompts`` function as the one reported below.

In [ ]:
def prepare_prompts(texts, prompt_template, tokenizer):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """
  pass

### Notes

1. You are free to modify the prompt format (**not its content**) as you like depending on your code implementation.

2. Note that the provided prompt has placeholders. You need to format the string to replace placeholders. Huggingface
   might have dedicated APIs for this.

# [Task 3 - 1.0 points] Inference

We are now ready to define the inference loop where we prompt the model with each pre-processed sample.

### Instructions

In order to get Task 3 points, we require you to:

* Write a ``generate_responses`` function as the one reported below.
* Write a ``process_response`` function as the one reported below.

In [ ]:
def generate_responses(model, tokenizer, prompt_examples):
  """
    This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card
      prompt_examples: pre-processed text samples

    Outputs:
      the generated responses, as strings, one per input example
  """
  pass

In [ ]:
def process_response(response):
  """
    This function takes a textual response generated by the LLM
    and processes it to map the response to a multi-label vector.

    Inputs:
      response: generated response from LLM

    Outputs:
      a 3-dimensional binary vector, in the order
      [credibility, emotion, logic].

      For instance:
        'logic'               -> [0, 0, 1]
        'credibility, emotion'-> [1, 1, 0]
        'none'                -> [0, 0, 0]

      Return None when the response mentions no category and does
      not say none: that is a failed response, see Task 4.
  """
  pass

## Notes

1. The response is free text: models will answer ``logic``, ``Logic.``, ``fallacy of logic``, or write a paragraph
   before committing to an answer. Decide how permissive your parser is and **write your rule in the report**, since
   it changes the numbers.

2. A response naming no category at all is **not** the same as a response saying ``none``. The first is a failure to
   follow instructions, the second is a prediction. Keep them apart.

3. According to our tests, it should take you ~10 mins to perform full inference on 156 samples on Colab.

# [Task 4 - 0.5 points] Metrics

In order to evaluate selected LLMs, we need to compute performance metrics.

We compute **macro F1-score** and the ratio of failed responses generated by models (**fail-ratio**).

That is, how frequent the LLM fails to follow instructions and provides incorrect responses that do not address the
classification task.

Since the task is multi-label, macro F1 is the average of the three **binary** F1-scores, one per category, each
computed on the positive class.

Failed responses are counted in the fail-ratio and treated as ``[0, 0, 0]`` when computing F1.

### Instructions

In order to get Task 4 points, we require you to:

* Write a ``compute_metrics`` function as the one reported below.
* Compute metrics for the two selected LLMs.
* Report the **per-category** F1-score alongside the macro average.

In [ ]:
def compute_metrics(responses, y_true):
  """
    This function takes the generated responses and the ground-truth
    labels and computes metrics. In particular, this function computes
    macro f1-score and fail-ratio metrics. It internally invokes
    `process_response`, since parsing is part of what we measure.

    Inputs:
      responses: the raw responses generated by the LLM, as strings
      y_true: ground-truth multi-label vectors

    Outputs:
      dictionary containing desired metrics
  """
  pass

### Note

A model that answers ``none`` to everything scores 0 macro F1 and a perfect fail-ratio. A model that answers
``credibility, logic, emotion`` to everything scores high recall and low precision. Neither is a good model, and only
reporting one number hides which one you have.

# [Task 5 - 1.0 points] Few-shot Inference

So far, we have tested models in a zero-shot fashion: we provide the input text to classify and instruct the model to
generate a response.

We are now interested in performing few-shot prompting to see the impact of providing demonstration examples.

To do so, we slightly change the prompt template as follows.

In [ ]:
prompt = [
    {
        'role': 'system',
        'content': 'You are an annotator for fallacy detection.'
    },
    {
        'role': 'user',
        'content': """Your task is to detect fallacies in the input text.
         A fallacy is an argument whose premises do not entail its
         conclusion. Classify the input text according to which of the
         following three categories of fallacy it contains.

         Below you find the category definitions:
         Credibility: the argument relies on who is speaking, or on who
         else agrees, instead of on evidence.
         Logic: the reasoning is broken, the conclusion does not follow
         from the premises.
         Emotion: the argument works on the feelings of the reader
         instead of on the merits of the claim.

         A text may contain more than one category, or none at all.

         Respond only by writing the categories that apply, separated by
         a comma, among the following: credibility, logic, emotion.
         Respond only by writing none if the text contains no fallacy.

        EXAMPLES: {examples}

        TEXT: {text}

        ANSWER:
        """
    }
]

The new prompt template reports some demonstration examples to instruct the model.

Generally, we provide an equal number of demonstrations per class as shown in the example below.

In [ ]:
prompt = [
    {
        'role': 'system',
        'content': 'You are an annotator for fallacy detection.'
    },
    {
        'role': 'user',
        'content': """... same instructions as above ...

         EXAMPLES:
         TEXT: **example 1**
         ANSWER: logic
         TEXT: **example 2**
         ANSWER: credibility, emotion
         TEXT: **example 3**
         ANSWER: none

         TEXT: {text}

        ANSWER:
        """
    }
]

## Instructions

In order to get Task 5 points, we require you to:

- Load ``demonstrations.csv`` and encode it into a ``pandas.DataFrame`` object.
- Define a ``build_few_shot_demonstrations`` function as the one reported below.
- Modify ``prepare_prompts`` to support demonstrations.
- Perform few-shot inference as in Task 3.
- Compute metrics as in Task 4.

In [ ]:
def build_few_shot_demonstrations(demonstrations, num_per_class=2):
  """
    Inputs:
      demonstrations: DataFrame wrapping demonstrations.csv
      num_per_class: number of demonstrations per class

    Outputs:
      list of demonstrations to inject into the prompt template.
  """
  pass

## Notes

1. You are free to pick any value for ``num_per_class``.

2. Demonstrations are multi-label too. Decide whether "one demonstration per class" means one text **containing**
   that category, or one text carrying **only** that category, and say which you chose. Do not forget texts with no
   fallacy: a model that never sees ``none`` will rarely answer it.

3. According to our tests, few-shot prompting increases inference time by some minutes (we experimented with
   ``num_per_class`` $\in [2, 4]$). MAFALDA texts are long, so keep an eye on the context window.

# [Task 6 - 1.0 points] Error Analysis

We are now interested in evaluating model responses and comparing their performance.

This analysis helps us in understanding

- Classification task performance gap: are the models good at this task?
- Generation quality: which kind of responses do models generate?
- Errors: which kind of mistakes do models do?

### Instructions

In order to get Task 6 points, we require you to:

* Compare classification performance of selected LLMs in a Table.
* Compute a confusion matrix **per category** for selected LLMs.
* Briefly summarize your observations on generated responses.

### Suggestions

Some questions worth answering:

* Do the models over-predict? Compare how many categories they assign per text against the 0.94 of the gold standard.
* Is `emotion` harder than `logic`, and is that the task or the 28 test texts that carry it?
* Do models catch the fallacy-free texts, or do they find a fallacy in everything?
* Do few-shot demonstrations fix the failures, or only the formatting of the answers?

# [Task 7 - 1.0 points] Report

Wrap up your experiment in a short report (up to 2 pages).

### Instructions

* Use the NLP course report template.
* Summarize each task in the report following the provided template.

### Recommendations

The report is not a copy-paste of graphs, tables, and command outputs.

* Summarize classification performance in Table format.
* **Do not** report command outputs or screenshots.
* The error analysis section should summarize your findings.

# Submission

* **Submit** your report in PDF format.
* **Submit** your python notebook.
* Make sure your notebook is **well organized**, with no temporary code, commented sections, tests, etc...

# FAQ

Please check this frequently asked questions before contacting us.

### Model cards

You can pick any open-source model card you like.

We recommend starting from those reported in this assignment.

### Implementation

Everything can be done via ``transformers`` APIs.

However, you are free to test frameworks, such as [LangChain](https://www.langchain.com/),
[LlamaIndex](https://www.llamaindex.ai/), provided that you correctly address task instructions.

### Task Performance

The task is challenging and zero-shot prompting may show relatively low performance depending on the chosen model.

The MAFALDA authors report that LLMs are still far from human performance on this benchmark, so low scores are an
expected result to be analysed, not a bug to be hidden.

### Prompt Template

Do not change the provided prompt template.

You are only allowed to change it in case of a possible extension.

### Optimizations

Any kind of code optimization (e.g., speedup model inference or reduce computational cost) is more than welcome!

### Bonus Points

0.5 bonus points are arbitrarily assigned based on significant contributions such as:

- Outstanding error analysis
- Masterclass code organization
- **Level 2 fallacy detection**: repeat the experiment on the ``labels_level2`` column, the 23 fine-grained fallacy
  types of MAFALDA. Beware that some of them appear in a handful of texts.
- Perform prompt tuning

Note that bonus points are only assigned if all task points are attributed (i.e., 6/6).

### Dataset Reference

Chadi Helwe, Tom Calamai, Pierre-Henri Paris, Chloé Clavel, Fabian Suchanek. 2024.
*MAFALDA: A Benchmark and Comprehensive Study of Fallacy Detection and Classification*. NAACL 2024.

The corpus is released under CC BY-SA 4.0.

# The End